# In Class Activity April 14th, 2026

In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [optuna]2m3/4 [optuna]]


### Importing libraries, preparing data, initial EDA

In [17]:
# importing libraries (feel free to add more if you want to explore other things)
import numpy as np
import pandas as pd
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
import optuna


In [3]:
# importing data
adult = pd.read_csv('/Users/christinewu/Desktop/academic resources/Advanced Machine Learning/In-Class Assignment Files/adult.csv')
adult.head(20)

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K
5,34,Private,198693,10th,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,<=50K
6,29,?,227026,HS-grad,9,Never-married,?,Unmarried,Black,Male,0,0,40,United-States,<=50K
7,63,Self-emp-not-inc,104626,Prof-school,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,>50K
8,24,Private,369667,Some-college,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,<=50K
9,55,Private,104996,7th-8th,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,<=50K


In [4]:
# initial EDA with sweetviz
report = sv.analyze(adult)
report.show_html('sweet_report.html')

# you are welcome to replace this cell with your own EDA, but make sure to include
# some visualizations and insights about the data


                                             |          | [  0%]   00:00 -> (? left)

Report sweet_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


### In the markdown cell below describe what you learned from your EDA and how it will inform your modeling decisions





In this dataset, featue types are a mix of demographics, financial variables, and categorical socio-economic variables. Some variables show skewness, and there are large amounts of numerical data. Many of the categorical variables have low numbers of classes (ex: gender), but some have lots of categories (such as country). Overall, the dataset is clean with no missing data, but categorical heavy which tells us that encoding variables will be key to building models. The next best action will be to remove some small number of duplicates within the data, encode the categorical variables, scale the numeric features, and check class balance of the target variable.

### Data Preprocessing (minimal) and Baseline Model

In [5]:
# data cleaning and preprocessing

# changing ? to NaN
adult = adult.replace('?', np.nan)

#education and education num are redundant, so we can drop one of them
adult = adult.drop('education', axis=1)

# target variable is income with 2 levels, so we can encode it as 0 and 1
adult['income'] = adult['income'].apply(lambda x: 1 if x == '>50K' else 0)

# change dtype categorical variables to category
categorical_cols = adult.select_dtypes(include='object').columns
adult[categorical_cols] = adult[categorical_cols].astype('category')


adult.head(20)

,age,workclass,fnlwgt,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,0
1,38,Private,89814,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,0
2,28,Local-gov,336951,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,1
3,44,Private,160323,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,1
4,18,NaN,103497,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,0
5,34,Private,198693,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,0
6,29,NaN,227026,9,Never-married,NaN,Unmarried,Black,Male,0,0,40,United-States,0
7,63,Self-emp-not-inc,104626,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,1
8,24,Private,369667,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,0
9,55,Private,104996,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,0


In [6]:
# defining features and target variable
X = adult.drop('income', axis=1)
y = adult['income']

# splitting data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, 
                                                    random_state=42, stratify=y)

In [7]:
# building xgboost default model and evaluating with stratified k-fold cross validation
xgb_cv = XGBClassifier(enable_categorical=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(xgb_cv, X, y, cv=skf, scoring='f1')
print(f'Cross-validated F1 scores: {cv_scores}')
print(f'Mean F1 score: {cv_scores.mean()}') 


Cross-validated F1 scores: [0.70680507 0.70892566 0.70898981 0.72424942 0.71086556]
Mean F1 score: 0.7119671046056588


### Use the markdown cell below to describe your baseline model performance and how you will try to improve it

With an f1-score of around 0.7, the model performed solid. Performance showed low variance across folds, so there is a good balance between precision and recall, but still has room to improve. How I would improve this model is by using hyperparameter tuning, since default XGBoost is not as optimal. I will focus on max_depth, learning_rate, and n_estimators, and also try out using gamma. I will also use Optuna. Additionally, I can try binning age group and creating interaction features to feature engineer the dataset.

### Model feature exploration

In the code cell below, explore different features of XGBoost and how they work (e.g. scale_pos_weight, max_depth, learning_rate).
Use stratified k-fold cross or repeated stratified k-fold cross validation with your model building. 
You should explore at least 3 different features of XGBoost.
Identify the model that performs best.

In [9]:
# exploring XGBoost features with stratified k-fold cross validation

neg = (y == 0).sum()
pos = (y == 1).sum()
scale_pos_weight_value = neg / pos

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.03, 0.1, 0.2],
    'scale_pos_weight': [1, scale_pos_weight_value]
}

results = []

for max_depth in param_grid['max_depth']:
    for learning_rate in param_grid['learning_rate']:
        for scale_pos_weight in param_grid['scale_pos_weight']:
            model = XGBClassifier(
                enable_categorical=True,
                random_state=42,
                eval_metric='logloss',
                n_estimators=300,
                max_depth=max_depth,
                learning_rate=learning_rate,
                scale_pos_weight=scale_pos_weight
            )
            
            scores = cross_val_score(model, X, y, cv=skf, scoring='f1', n_jobs=-1)
            
            results.append({
                'max_depth': max_depth,
                'learning_rate': learning_rate,
                'scale_pos_weight': scale_pos_weight,
                'mean_f1': scores.mean(),
                'std_f1': scores.std()
            })

results_df = pd.DataFrame(results).sort_values(by='mean_f1', ascending=False).reset_index(drop=True)

print("Top models:")
print(results_df)

best_params = results_df.iloc[0]

print("\nBest model parameters:")
print(best_params)

best_model = XGBClassifier(
    enable_categorical=True,
    random_state=42,
    eval_metric='logloss',
    n_estimators=300,
    max_depth=int(best_params['max_depth']),
    learning_rate=float(best_params['learning_rate']),
    scale_pos_weight=float(best_params['scale_pos_weight'])
)

best_scores = cross_val_score(best_model, X, y, cv=skf, scoring='f1', n_jobs=-1)

print(f"\nBest model CV F1 scores: {best_scores}")
print(f"Best model mean F1: {best_scores.mean()}")

Top models:
    max_depth  learning_rate  scale_pos_weight   mean_f1    std_f1
0           7           0.10          3.179173  0.717687  0.004223
1           5           0.10          3.179173  0.716445  0.006266
2           5           0.20          3.179173  0.716336  0.005747
3           5           0.10          1.000000  0.715883  0.006555
4           3           0.20          3.179173  0.715506  0.005569
5           7           0.03          3.179173  0.715449  0.005987
6           3           0.20          1.000000  0.714195  0.008394
7           7           0.20          3.179173  0.713621  0.007195
8           3           0.10          3.179173  0.713554  0.004792
9           7           0.03          1.000000  0.712452  0.006603
10          5           0.20          1.000000  0.712342  0.005542
11          7           0.10          1.000000  0.711924  0.005518
12          3           0.10          1.000000  0.711583  0.007543
13          5           0.03          3.179173  0.

### Tuning with GridSearchCV

Use the code cell below to set up your parameter grid and run GridSearchCV with your preferred model from above. You should tune 4-5 hyperparameters utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [16]:
# tuning the preferred XGBoost model with GridSearchCV

param_grid = {
    'max_depth': [5, 7],
    'learning_rate': [0.05, 0.1],
    'scale_pos_weight': [3.0],
    'n_estimators': [200, 300],
    'min_child_weight': [1, 3]
}

xgb_grid = GridSearchCV(
    estimator=XGBClassifier(
        random_state=42,
        enable_categorical=True,
        eval_metric='logloss',
        tree_method='hist'
    ),
    param_grid=param_grid,
    scoring='f1',
    cv=skf,
    n_jobs=-1,
    verbose=1
)

xgb_grid.fit(X_train, y_train)

print(f'Best parameters from GridSearchCV: {xgb_grid.best_params_}')
print(f'Best F1 score from GridSearchCV: {xgb_grid.best_score_}')

# build final model using best parameters
xgb_grid_best = XGBClassifier(
    random_state=42,
    enable_categorical=True,
    eval_metric='logloss',
    tree_method='hist',
    max_depth=xgb_grid.best_params_['max_depth'],
    learning_rate=xgb_grid.best_params_['learning_rate'],
    scale_pos_weight=xgb_grid.best_params_['scale_pos_weight'],
    n_estimators=xgb_grid.best_params_['n_estimators'],
    min_child_weight=xgb_grid.best_params_['min_child_weight']
)

xgb_grid_best.fit(X_train, y_train)
y_pred_grid = xgb_grid_best.predict(X_test)

print(f'\nTest F1 score: {f1_score(y_test, y_pred_grid)}')
print(f'\nClassification report for GridSearchCV-tuned model:\n{classification_report(y_test, y_pred_grid)}')
print(f'Confusion matrix:\n{confusion_matrix(y_test, y_pred_grid)}')

Fitting 5 folds for each of 16 candidates, totalling 80 fits


Best parameters from GridSearchCV: {'learning_rate': 0.1, 'max_depth': 7, 'min_child_weight': 1, 'n_estimators': 300, 'scale_pos_weight': 3.0}
Best F1 score from GridSearchCV: 0.7178848787005881

Test F1 score: 0.7205636936769887

Classification report for GridSearchCV-tuned model:
              precision    recall  f1-score   support

           0       0.94      0.85      0.89      7431
           1       0.64      0.83      0.72      2338

    accuracy                           0.85      9769
   macro avg       0.79      0.84      0.81      9769
weighted avg       0.87      0.85      0.85      9769

Confusion matrix:
[[6319 1112]
 [ 395 1943]]


### Tuning with RandomizedSearchCV

Using the code cell below as a starting point, tune your preferred model from above. Tune the same 4-5 hyperparameters from above utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [ ]:
# tuning xgboost classifier with RandomizedSearchCV (tune more parameters than shown here)
param_dist = {
    'scale_pos_weight': np.linspace(1.0, 10.0, num=10),
    'max_depth': np.arange(3, 11),
    'learning_rate': np.linspace(0.01, 0.3, num=10)
}

# replace this placeholder model with your preferred model from above

xgb_random = RandomizedSearchCV(XGBClassifier(random_state=42, enable_categorical=True),
                                param_distributions=param_dist, n_iter=20, cv=skf, scoring='f1', random_state=42)
xgb_random.fit(X_train, y_train)
print(f'Best parameters from RandomizedSearchCV: {xgb_random.best_params_}')
print(f'Best F1 score from RandomizedSearchCV: {xgb_random.best_score_}')   

# build your preferred model from above using best parameters from your RandomizedSearchCV
# and evaluate on the test set

xgb_random_best = XGBClassifier(random_state=42, scale_pos_weight=xgb_random.best_params_['scale_pos_weight'], 
                                max_depth=xgb_random.best_params_['max_depth'], 
                                learning_rate=xgb_random.best_params_['learning_rate'], 
                                enable_categorical=True)
xgb_random_best.fit(X_train, y_train)
y_pred_random = xgb_random_best.predict(X_test)
print(f'Classification report for RandomizedSearchCV-tuned model:\n{classification_report(y_test, y_pred_random)}')


In [14]:
# tuning the preferred XGBoost model with RandomizedSearchCV

from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import f1_score, classification_report, confusion_matrix

param_dist = {
    'max_depth': np.arange(5, 10),
    'learning_rate': np.linspace(0.05, 0.2, num=10),
    'scale_pos_weight': np.linspace(2.0, 4.5, num=10),
    'n_estimators': np.arange(100, 351, 50),
    'min_child_weight': np.arange(1, 6)
}

xgb_random = RandomizedSearchCV(
    XGBClassifier(
        random_state=42,
        enable_categorical=True,
        eval_metric='logloss',
        tree_method='hist'
    ),
    param_distributions=param_dist,
    n_iter=20,
    cv=skf,
    scoring='f1',
    random_state=42,
    n_jobs=-1
)

xgb_random.fit(X_train, y_train)

print(f'Best parameters from RandomizedSearchCV: {xgb_random.best_params_}')
print(f'Best F1 score from RandomizedSearchCV: {xgb_random.best_score_}')

# build final preferred model using best parameters
xgb_random_best = XGBClassifier(
    random_state=42,
    enable_categorical=True,
    eval_metric='logloss',
    tree_method='hist',
    max_depth=xgb_random.best_params_['max_depth'],
    learning_rate=xgb_random.best_params_['learning_rate'],
    scale_pos_weight=xgb_random.best_params_['scale_pos_weight'],
    n_estimators=xgb_random.best_params_['n_estimators'],
    min_child_weight=xgb_random.best_params_['min_child_weight']
)

xgb_random_best.fit(X_train, y_train)
y_pred_random = xgb_random_best.predict(X_test)

print(f'\nTest F1 score: {f1_score(y_test, y_pred_random)}')
print(f'\nClassification report for RandomizedSearchCV-tuned model:\n{classification_report(y_test, y_pred_random)}')
print(f'Confusion matrix:\n{confusion_matrix(y_test, y_pred_random)}')

Best parameters from RandomizedSearchCV: {'scale_pos_weight': np.float64(2.0), 'n_estimators': np.int64(200), 'min_child_weight': np.int64(5), 'max_depth': np.int64(7), 'learning_rate': np.float64(0.05)}
Best F1 score from RandomizedSearchCV: 0.7259448881404686

Test F1 score: 0.7246717617087988

Classification report for RandomizedSearchCV-tuned model:
              precision    recall  f1-score   support

           0       0.93      0.88      0.90      7431
           1       0.67      0.79      0.72      2338

    accuracy                           0.86      9769
   macro avg       0.80      0.83      0.81      9769
weighted avg       0.87      0.86      0.86      9769

Confusion matrix:
[[6515  916]
 [ 489 1849]]


### Tuning with Optuna

Using the code cell below as a starting point, tune your preferred model from above. You should tune the same 4-5 parameters as above using cross validation. Train a final model using the best hyperparameters and report your model performance.

In [ ]:
# tuning with Optuna (tune more parameters than shown here)
def objective(trial):
    scale_pos_weight = trial.suggest_float('scale_pos_weight', 1.0, 10.0)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)
    
    # replace this placeholder model with your preferred model from above
    
    xgb_optuna = XGBClassifier(random_state=42, scale_pos_weight=scale_pos_weight, 
                               max_depth=max_depth,  learning_rate=learning_rate, enable_categorical=True)
    
    cv_scores = cross_val_score(xgb_optuna, X, y, cv=skf, scoring='f1')
    return cv_scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20, show_progress_bar=True)

print(f'Best parameters from Optuna: {study.best_params}')
print(f'Best F1 score from Optuna: {study.best_value}')

# build preferred model from above with best parameters from Optuna and evaluate on the test set
xgb_optuna_best = XGBClassifier(random_state=42, scale_pos_weight=study.best_params['scale_pos_weight'], 
                                  max_depth=study.best_params['max_depth'], 
                                  learning_rate=study.best_params['learning_rate'], 
                                  enable_categorical=True)
xgb_optuna_best.fit(X_train, y_train)
y_pred_optuna = xgb_optuna_best.predict(X_test)
print(f'Classification report for Optuna-tuned model:\n{classification_report(y_test, y_pred_optuna)}')


In [13]:
# tuning the preferred XGBoost model with Optuna

def objective(trial):
    max_depth = trial.suggest_int('max_depth', 5, 9)
    learning_rate = trial.suggest_float('learning_rate', 0.05, 0.2)
    scale_pos_weight = trial.suggest_float('scale_pos_weight', 2.0, 4.5)
    n_estimators = trial.suggest_int('n_estimators', 100, 300, step=50)
    min_child_weight = trial.suggest_int('min_child_weight', 1, 5)

    xgb_optuna = XGBClassifier(
        random_state=42,
        enable_categorical=True,
        eval_metric='logloss',
        tree_method='hist',
        max_depth=max_depth,
        learning_rate=learning_rate,
        scale_pos_weight=scale_pos_weight,
        n_estimators=n_estimators,
        min_child_weight=min_child_weight
    )

    cv_scores = cross_val_score(xgb_optuna, X, y, cv=skf, scoring='f1', n_jobs=-1)
    return cv_scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20, show_progress_bar=True)

print(f'Best parameters from Optuna: {study.best_params}')
print(f'Best F1 score from Optuna: {study.best_value}')

# train final model using best Optuna hyperparameters
xgb_optuna_best = XGBClassifier(
    random_state=42,
    enable_categorical=True,
    eval_metric='logloss',
    tree_method='hist',
    max_depth=study.best_params['max_depth'],
    learning_rate=study.best_params['learning_rate'],
    scale_pos_weight=study.best_params['scale_pos_weight'],
    n_estimators=study.best_params['n_estimators'],
    min_child_weight=study.best_params['min_child_weight']
)

xgb_optuna_best.fit(X_train, y_train)
y_pred_optuna = xgb_optuna_best.predict(X_test)

print(f'\nTest F1 score: {f1_score(y_test, y_pred_optuna)}')
print(f'\nClassification report for Optuna-tuned model:\n{classification_report(y_test, y_pred_optuna)}')
print(f'Confusion matrix:\n{confusion_matrix(y_test, y_pred_optuna)}')

[I 2026-04-14 13:38:02,626] A new study created in memory with name: no-name-c5dcfaa5-fc3d-4097-a7fb-5e191aad1ac0


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-04-14 13:38:05,960] Trial 0 finished with value: 0.7135051786424167 and parameters: {'max_depth': 8, 'learning_rate': 0.1771633916535064, 'scale_pos_weight': 3.315324583943305, 'n_estimators': 250, 'min_child_weight': 3}. Best is trial 0 with value: 0.7135051786424167.
[I 2026-04-14 13:38:07,884] Trial 1 finished with value: 0.6999493010775233 and parameters: {'max_depth': 7, 'learning_rate': 0.05847561361659139, 'scale_pos_weight': 4.004447805740516, 'n_estimators': 100, 'min_child_weight': 1}. Best is trial 0 with value: 0.7135051786424167.
[I 2026-04-14 13:38:11,053] Trial 2 finished with value: 0.7071987136791165 and parameters: {'max_depth': 8, 'learning_rate': 0.17936169245426264, 'scale_pos_weight': 4.412572861443203, 'n_estimators': 200, 'min_child_weight': 4}. Best is trial 0 with value: 0.7135051786424167.
[I 2026-04-14 13:38:15,233] Trial 3 finished with value: 0.7161459739889546 and parameters: {'max_depth': 8, 'learning_rate': 0.07137186955489136, 'scale_pos_weight

### Tuning results

In the markdown cell below describe your experience tuning with the different methods. Which produced the best results? Which do you prefer?


Overall, the model tuned with Optuna produced the best results, and was also the fastest to tune. The GridSearchCV took forever to tune 1215 fits, and the RandomizedSearchCV was faster than the GridSearchCV but did not yield an f1-score as high as the Optuna. Overall, I prefer Optuna.